# Julián Serrano Chacón
# Mika Rodríguez Castro
  
# Práctica 3

In [15]:
from importlib import reload
import numpy as np
import matplotlib.pyplot as plt
import sounddevice as sd
import time # para medir tiempos de ejecución
import soundfile as sf     
from ipywidgets import interact
from ipywidgets import fixed
import sys
import time
import scipy.signal as sg
import synthFM as sfm
reload(sfm)
reload(sfm.osc)
from tkinter import *
from consts import *
from tkinter import *
from oscFM import *
from adsr import *

# graficos en el notebook
%matplotlib inline

In [10]:
root=Tk()
def key_pressed(event):
    key = event.char
    if key in teclas:
        index = teclas.index(key) # sacamos posición en el string notas
        nota = notas[index]
        pitch = pitchs[index]
        print(f'tecla {key} nota {nota} pitch {pitch}')
    elif key == '-':
        print('note off')

    root.bind("<Key>",key_pressed)
    root.mainloop()

In [12]:
%%writefile consts.py

# mapeo de teclas del ordenador a notas en el piano
# utilizamos '.' para los sostenidos
teclas = "zsxdcvgbhnjmq2w3er5t6y7u"  # 2 de teclas filas 
notas =  "C.D.EF.G.A.Bc.d.ef.g.a.b"  # mapeadas a 2 octavas
#         octava baja||octava alta


# frecuencias de las notas asociadas a las teclas del teclado
# partimos del la=220Hz y generamos frecuencias de escala temperada
pitchs = [ 220*2.0**(i/12.0) for i in range(len(teclas))] 

# frecuencias asociadas a las notas midi de 0 a 127
# El LA central es la nota midi 70 y su frecuencia es 440
# construimos hacia abajo y hacia arriba el resto de notas
freqsMidi = [ 440*2.0**(i/12.0) for i in range(-69,59)]


SRATE = 48000 # Sample rate, para todo el notebook

CHUNK = 1024


Overwriting consts.py


#### Ejercicio 1 (Obligatorio)  
Utilizar el controlador de teclado propuesto para implementar un pequeño instrumento monofónico que utilize el sintetizador FM visto en clase como generador de señal.  


In [16]:
synths = sfm.SynthFM()
input = None

def callback(outdata, frames, time, status):
    if status: print(status)
    # si hay generador de señal conectado, pedimos el siguiente bloque
    if input:    
        s = input.next()
        s = np.float32(s)
    # si no, generamos silencio    
    else:
        s = np.zeros(CHUNK,dtype=np.float32)

    outdata[:] = s.reshape(-1, 1)

# stream de salida con callBack
stream = sd.OutputStream(samplerate=SRATE, callback=callback, blocksize=CHUNK)
stream.start()

root=Tk()

# Caja de texto
text = Text(root,height=6,width=60)
text.pack(side=BOTTOM)
text.insert(INSERT,"Press keys\n")

def key_pressed(event):
    global synths
    global input
    key = event.char
    if key in teclas:
        index = teclas.index(key) # sacamos posición en el string notas
        synths = sfm.SynthFM(pitchs[index],beta=0.5)
        input = synths
    elif key == '-':
        synths.noteOff()

text.bind("<Key>",key_pressed)
root.mainloop()

# limpieza..
stream.stop()
stream.close()

TypeError: OscFM.__init__() got multiple values for argument 'beta'

#### Ejercicio 2
Secuenciar la canción de cumpleaños feliz de la hoja anterior utilizando el sintetizador FM visto
en clase. Para ello se lanzaran secuencialmente las notas de la canción, se dejarán sonar durante el
tiempo estipulado y se apagarán con el método noteOff() del sintetizador.

In [17]:
synth = sfm.SynthFM()
rel = 0.1

pitches = {
     'C': 523.251, 'D' : 587.33, 'E' : 659.255, 'F' : 698.456, 'G' : 783.991, 'A': 880.0, 'B' : 987.767,
         'c' : 523.251/2, 'd' : 587.33/2, 'e' : 659.255/2, 'f' : 698.456/2, 'g' : 783.991/2, 'a' : 880.0/2, 'b': 987.767/2, ' ' : 0
    }

happyB = [('G', 0.5),  ('G', 0.5), ('A', 1.0), ('G', 1.0), ('c', 1.0), ('B', 2.0), ('G', 0.5),
           ('G', 0.5), ('A', 1.0), ('G', 1.0), ('d',1.0),('c', 2.0), ('G', 0.5), 
          ('G', 0.5), ('g', 1.0), ('e', 1.0), ('c',1.0), ('B',1.0),('A',1.0), ('f',0.5),
           ('f',0.5), ('e',1.0), ('c',1.0), ('d',1.0), ('c',2.0)]

def callback(outdata, frames, time, status):
    if status: print(status)
    # si hay generador de señal conectado, pedimos el siguiente bloque
    if synth: 
        s = synth.next()
        s = np.float32(s)
    # si no, generamos silencio    
    else:
        s = np.zeros(CHUNK,dtype=np.float32)

    outdata[:] = s.reshape(-1, 1)

# stream de salida con callBack
stream = sd.OutputStream(samplerate=SRATE, callback=callback, blocksize=CHUNK)
stream.start()

for t in happyB:
    synth = sfm.SynthFM(pitches[t[0]],release=rel)
    time.sleep(t[1]) # puede que haga la ejecución ininterrumpible, buscar alternativas
    synth.noteOff()
    time.sleep(rel)



TypeError: OscFM.__init__() got multiple values for argument 'beta'

In [ ]:
stream.stop()

#### Ejercicio 3
Extender el sintetizador FM presentado en clase de modo que pueda trabajar con distintos tipos
de onda (sinosuoidal, cuadrada, triangular, diente de sierra) tanto para la portadora fc como para
la moduladora fm.  
Puede hacerse una pequeña interfaz gráfica para probar el nuevo instrumento.

In [ ]:
signal = sfm.SynthFM(fc=440)
shapes = [
    'sin',
    'square',
    'sawtooth',
    'triangle'
]

def callback(outdata, frames, time, status):
    if status: print(status)
    # si hay generador de señal conectado, pedimos el siguiente bloque
    if signal:
        s = signal.next()
        s = np.float32(s)
    # si no, generamos silencio    
    else:
        s = np.zeros(CHUNK,dtype=np.float32)

    outdata[:] = s.reshape(-1, 1)

# stream de salida con callBack
stream = sd.OutputStream(samplerate=SRATE, callback=callback, blocksize=CHUNK)
stream.start()

# 0 = sin, 1 = square, 2 = sawtooth, 3 = triangle
def fmCoefCtrl(fmShape):
    signal.setFmShape(shapes[fmShape])
interact(fmCoefCtrl, fmShape=(0,3,1))

def fcCoefCtrl(fcShape):
    signal.setFcShape(shapes[fcShape])
interact(fcCoefCtrl, fcShape=(0,3,1))

def pauseCtrl(pause):
    if pause==True:
        stream.stop()        
    else:
        stream.start()   
# lo interpreta como un checkbox
interact(pauseCtrl, pause=False)       

def stopCtrl(stop):
    if stop==True:
        stream.stop()
        stream.close()
interact(stopCtrl, stop=False) 
